In [ ]:
"""
---------------------------------------------------------
STATISTICAL SPELL CHECKER USING NOISY CHANNEL MODEL
---------------------------------------------------------
"""

import json
import re
import time
from collections import Counter, defaultdict
from nltk.metrics import edit_distance


# -----------------------------------------------------
# 1. LOAD DATASET
# -----------------------------------------------------

file_path = r"D:\dev.jsonl"

data = []

with open(file_path, "r", encoding="utf-8") as f:
    for line in f:
        data.append(json.loads(line))

print("Total records:", len(data))


# -----------------------------------------------------
# 2. CREATE CORPUS
# -----------------------------------------------------

corpus = []

for item in data:
    if "facts" in item:
        corpus.append(item["facts"])

flat_corpus = []

for fact_list in corpus:
    flat_corpus.extend(fact_list)

print("Total sentences:", len(flat_corpus))


# -----------------------------------------------------
# 3. CLEAN TEXT
# -----------------------------------------------------

def clean_text(text):

    text = text.lower()
    text = re.sub(r'\d+', '', text)
    text = re.sub(r'[^a-z\s]', ' ', text)   # FIXED punctuation cleaning
    text = re.sub(r'\s+', ' ', text).strip()

    return text


corpus_clean = [clean_text(sentence) for sentence in flat_corpus]


# -----------------------------------------------------
# 4. TOKENIZATION
# -----------------------------------------------------

tokens = []

for sentence in corpus_clean:
    tokens.extend(sentence.split())

print("Total tokens:", len(tokens))


# -----------------------------------------------------
# 5. VOCABULARY AND FREQUENCY
# -----------------------------------------------------

word_freq = Counter(tokens)

print("Original vocabulary:", len(word_freq))

threshold = 2

filtered_tokens = [word for word in tokens if word_freq[word] >= threshold]

vocab = set(filtered_tokens)

print("Filtered vocabulary:", len(vocab))


# -----------------------------------------------------
# 6. LANGUAGE MODELS
# -----------------------------------------------------

total_words = len(filtered_tokens)

unigram_counts = Counter(filtered_tokens)

V = len(vocab)

# FIXED: add Laplace smoothing
def unigram_prob(word):
    return (unigram_counts[word] + 1) / (total_words + V)


# ---------------------------
# BIGRAM MODEL (FIXED)
# ---------------------------

bigram_counts = defaultdict(int)

for sentence in corpus_clean:

    words = sentence.split()

    for i in range(len(words) - 1):

        w1 = words[i]
        w2 = words[i + 1]

        if w1 in vocab and w2 in vocab:
            bigram_counts[(w1, w2)] += 1


# ---------------------------
# TRIGRAM MODEL (FIXED)
# ---------------------------

trigram_counts = defaultdict(int)

for sentence in corpus_clean:

    words = sentence.split()

    for i in range(len(words) - 2):

        w1 = words[i]
        w2 = words[i + 1]
        w3 = words[i + 2]

        if w1 in vocab and w2 in vocab and w3 in vocab:
            trigram_counts[(w1, w2, w3)] += 1


# -----------------------------------------------------
# 7. LAPLACE SMOOTHING
# -----------------------------------------------------

def bigram_prob(w1, w2):

    return (bigram_counts[(w1, w2)] + 1) / (unigram_counts[w1] + V)


def trigram_prob(w1, w2, w3):

    return (trigram_counts[(w1, w2, w3)] + 1) / (bigram_counts[(w1, w2)] + V)


# -----------------------------------------------------
# 8. ERROR MODEL
# -----------------------------------------------------

def get_candidates(word):

    candidates = []

    for v in vocab:

        # FIXED crash condition
        if word and v and v[0] == word[0]:

            if edit_distance(word, v) <= 2:

                candidates.append(v)

    return candidates


def error_prob(typo, word):

    d = edit_distance(typo, word)

    return 1 / (d + 1)


# -----------------------------------------------------
# 9. SPELL CORRECTION MODELS
# -----------------------------------------------------

def correct_word_unigram(word):

    candidates = get_candidates(word)

    if not candidates:
        return word

    best_word = word
    best_score = 0

    for c in candidates:

        p_language = unigram_prob(c)
        p_error = error_prob(word, c)

        score = p_language * p_error

        if score > best_score:
            best_score = score
            best_word = c

    return best_word


def correct_word_bigram(prev_word, word):

    candidates = get_candidates(word)

    if not candidates:
        return word

    best_word = word
    best_score = 0

    for c in candidates:

        p_language = bigram_prob(prev_word, c)
        p_error = error_prob(word, c)

        score = p_language * p_error

        if score > best_score:
            best_score = score
            best_word = c

    return best_word


def correct_word_trigram(prev1, prev2, word):

    candidates = get_candidates(word)

    if not candidates:
        return word

    best_word = word
    best_score = 0

    for c in candidates:

        p_language = trigram_prob(prev1, prev2, c)
        p_error = error_prob(word, c)

        score = p_language * p_error

        if score > best_score:
            best_score = score
            best_word = c

    return best_word


# -----------------------------------------------------
# 10. SENTENCE CORRECTION
# -----------------------------------------------------

def correct_sentence_unigram(sentence):

    words = sentence.lower().split()

    return " ".join(correct_word_unigram(w) for w in words)


def correct_sentence_bigram(sentence):

    words = sentence.lower().split()

    corrected = words.copy()

    for i in range(1, len(words)):

        corrected[i] = correct_word_bigram(corrected[i - 1], words[i])

    return " ".join(corrected)


def correct_sentence_trigram(sentence):

    words = sentence.lower().split()

    corrected = words.copy()

    for i in range(2, len(words)):

        corrected[i] = correct_word_trigram(corrected[i - 2], corrected[i - 1], words[i])

    return " ".join(corrected)


# -----------------------------------------------------
# 11. TEST SET
# -----------------------------------------------------

test_set = [
("the court helf that the defendnt was detaned",
 "the court held that the defendant was detained"),

("the juge gav the final verdct",
 "the judge gave the final verdict")
]


# -----------------------------------------------------
# 12. EVALUATION METRICS
# -----------------------------------------------------

def evaluate(model):

    TP = FP = FN = 0

    for typo, correct in test_set:

        predicted = model(typo)

        p_words = predicted.split()
        c_words = correct.split()
        t_words = typo.split()

        for p, c, t in zip(p_words, c_words, t_words):

            if p == c and t != c:
                TP += 1
            elif p != c and t != c:
                FN += 1
            elif p != c and t == c:
                FP += 1

    precision = TP / (TP + FP) if TP + FP > 0 else 0
    recall = TP / (TP + FN) if TP + FN > 0 else 0

    f1 = 0

    if precision + recall > 0:
        f1 = 2 * precision * recall / (precision + recall)

    return precision, recall, f1


# -----------------------------------------------------
# 13. RUN EXPERIMENTS
# -----------------------------------------------------

print("\nOriginal Test:")

test = "the court helf that the defendnt was detaned"

print("Original:", test)

print("Unigram:", correct_sentence_unigram(test))
print("Bigram:", correct_sentence_bigram(test))
print("Trigram:", correct_sentence_trigram(test))


# -----------------------------------------------------
# 14. TIME COMPLEXITY
# -----------------------------------------------------

sample = test

start = time.time()
correct_sentence_unigram(sample)
print("Unigram time:", time.time() - start)

start = time.time()
correct_sentence_bigram(sample)
print("Bigram time:", time.time() - start)

start = time.time()
correct_sentence_trigram(sample)
print("Trigram time:", time.time() - start)


# -----------------------------------------------------
# 15. EVALUATION
# -----------------------------------------------------

print("\nEvaluation Results")

print("Unigram:", evaluate(correct_sentence_unigram))
print("Bigram:", evaluate(correct_sentence_bigram))
print("Trigram:", evaluate(correct_sentence_trigram))

Total records: 1000
Total sentences: 24418
Total tokens: 1715709
Original vocabulary: 24358
Filtered vocabulary: 15528

Original Test:
Original: the court helf that the defendnt was detaned
Unigram: the court he the the defendant was detained
Bigram: the court held that the defendant was detained
Trigram: the court held that the defendant was detained
Unigram time: 0.2919306755065918
Bigram time: 0.3049466609954834
Trigram time: 0.18557047843933105

Evaluation Results
Unigram: (0.75, 0.5, 0.6)
Bigram: (1.0, 1.0, 1.0)
Trigram: (1.0, 0.6666666666666666, 0.8)
